# The Browser Object Model (BOM)

The BOM is the set of browser-provided objects that let JavaScript interact with the browser itself. Where the DOM handles page *content*, the BOM handles everything around it — windows, history, screen, and the current URL.

There is no single official BOM standard, but modern browsers implement effectively identical APIs. (Much of it has since been absorbed into the WHATWG HTML Living Standard, so "unstandardised" is less true than it used to be.)

Related: [[JS - Script Loading, defer and async]]

---

## 1. Hierarchy

`window` sits at the top. Every other browser object hangs off it, and because `window` *is* the global object in a browser, the prefix is optional.

```
                        [ window ]
                             │
   ┌──────────┬──────────┬───┴────────┬───────────┬──────────────┐
[screen]  [history]  [location]  [navigator]  [localStorage]  [document] ──▶ the DOM
```

```js
window.location.href === location.href;  // true — 'window.' is implicit
window === window.window;                // true — it points at itself
```

> **Node has no BOM.** `window`, `document`, and `location` do not exist there — the global object is `globalThis` (or the legacy `global`). Use `globalThis` when writing code that must run in both. This matters for your Express work: anything DOM- or BOM-touching is browser-only.

---

## 2. The `window` Object

Represents the current tab or frame. Controls sizing, scrolling, dialogs, and timers.

```js
window.innerWidth;   // viewport width in px, including scrollbar
window.innerHeight;
window.outerWidth;   // whole browser window, including chrome
window.scrollY;      // current vertical scroll position

window.open('https://example.com', '_blank');
window.close();      // only works on windows opened by script

setTimeout(fn, 1000);
setInterval(fn, 1000);
```

### Additions worth knowing

**`devicePixelRatio`** — CSS pixels to device pixels. `2` on a Retina display; needed for sharp canvas rendering.

**Scrolling APIs** are better than setting `scrollY`:

```js
window.scrollTo({ top: 0, behavior: 'smooth' });
element.scrollIntoView({ behavior: 'smooth', block: 'center' });
```

**Dialogs are blocking and best avoided.** `alert()`, `confirm()`, and `prompt()` freeze the entire page and are ignored or throttled in some contexts (cross-origin iframes, during unload).

**`window.name` is a footgun.** It's a real string property that persists across navigations in the same tab, which is why `this.name = name` in a constructor called without `new` silently clobbers it — see [[JS - The new Operator]].

**`requestAnimationFrame`** should replace `setInterval` for animation. It syncs to the display refresh and pauses in background tabs:

```js
function loop() {
  update();
  requestAnimationFrame(loop);
}
requestAnimationFrame(loop);
```

**Cross-frame messaging:**

```js
otherWindow.postMessage(data, 'https://trusted.example.com');
window.addEventListener('message', (e) => {
  if (e.origin !== 'https://trusted.example.com') return;  // always check
  handle(e.data);
});
```

---

## 3. The `location` Object

Information about the current URL, and the ability to navigate.

For `https://shop.example.com:8080/products/list?page=2&sort=asc#reviews`:

| Property | Value |
|---|---|
| `href` | the whole string |
| `protocol` | `"https:"` |
| `hostname` | `"shop.example.com"` |
| `host` | `"shop.example.com:8080"` |
| `port` | `"8080"` |
| `pathname` | `"/products/list"` |
| `search` | `"?page=2&sort=asc"` |
| `hash` | `"#reviews"` |
| `origin` | `"https://shop.example.com:8080"` |

### Navigation

```js
location.href = '/dashboard';   // navigate, adds a history entry
location.assign('/dashboard');  // identical
location.replace('/login');     // navigate WITHOUT a history entry
location.reload();              // refresh
```

`replace()` is the right choice for redirects after login or logout — otherwise the back button returns the user to a page they've moved past.

### Parsing query strings

Don't split `location.search` by hand:

```js
const params = new URLSearchParams(location.search);
params.get('page');           // "2"
params.has('sort');           // true
params.getAll('tag');         // handles repeated keys

const url = new URL(location.href);
url.searchParams.set('page', '3');
url.toString();
```

`URL` and `URLSearchParams` also exist in Node, so this knowledge transfers.

---

## 4. The `navigator` Object

Identity, state, and permissions for the user's browser and device.

```js
navigator.language;      // "en-GB"
navigator.languages;     // ["en-GB", "en", "hi"] — ordered preference
navigator.onLine;        // boolean
navigator.userAgent;     // browser identification string
```

**`userAgent` is unreliable.** It has been deliberately spoofed and frozen for privacy reasons, and browsers actively lie in it. Never branch on it — use feature detection instead:

```js
// Bad
if (navigator.userAgent.includes('Chrome')) { ... }

// Good
if ('IntersectionObserver' in window) { ... }
```

**`onLine` only means "has a network interface."** It cannot tell you whether the internet actually works — a captive portal reports `true`. Treat it as a hint, and pair it with the events:

```js
window.addEventListener('online',  () => ...);
window.addEventListener('offline', () => ...);
```

### Useful modern additions

```js
navigator.clipboard.writeText('copied');        // async, requires user gesture
navigator.geolocation.getCurrentPosition(cb);   // prompts for permission
navigator.share({ title, url });                // native share sheet, mobile
navigator.sendBeacon('/analytics', data);       // fire-and-forget on page unload
navigator.hardwareConcurrency;                  // logical CPU cores
```

`sendBeacon` is the correct way to send analytics during unload — a normal `fetch` gets cancelled when the page goes away.

---

## 5. The `history` Object

The current tab's session history.

```js
history.back();      // same as the back button
history.forward();
history.go(-2);      // back two entries
history.length;      // number of entries in this session
```

You cannot read the URLs in history — that would be a privacy leak. You can only move through it.

### The History API — how SPA routing works

```js
history.pushState({ page: 2 }, '', '/products?page=2');   // add an entry, no reload
history.replaceState({ page: 2 }, '', '/products');       // modify current entry

window.addEventListener('popstate', (e) => {
  render(e.state);   // fires on back/forward
});
```

This is the whole mechanism behind client-side routers. `pushState` changes the URL bar without a network request; `popstate` lets you re-render when the user hits back. Note `popstate` does **not** fire for `pushState` itself — only for user navigation.

---

## 6. The `screen` Object

Hardware display properties.

```js
screen.width;         // full display resolution
screen.height;
screen.availWidth;    // usable area, excluding OS taskbar/dock
screen.availHeight;
screen.colorDepth;    // bits per pixel, usually 24
screen.orientation.type;  // "portrait-primary" etc.
```

**Don't use `screen` for responsive layout.** It reports the physical monitor, not your window — a maximised browser on a 4K display and a narrow one report the same `screen.width`. Use `window.innerWidth`, or better, `matchMedia`:

```js
const mq = window.matchMedia('(max-width: 768px)');
mq.matches;                              // boolean now
mq.addEventListener('change', handler);  // reacts to resize
```

---

## 7. Storage (also part of the BOM)

Frequently omitted from BOM lists but sits on `window` alongside the rest.

| | Persists | Scope | Size |
|---|---|---|---|
| `localStorage` | Until cleared | Per origin | ~5–10 MB |
| `sessionStorage` | Until tab closes | Per tab | ~5–10 MB |
| `document.cookie` | Per expiry | Sent with every request | ~4 KB |

```js
localStorage.setItem('theme', 'dark');
localStorage.getItem('theme');     // "dark"
localStorage.removeItem('theme');
```

Strings only — `JSON.stringify` on the way in, `JSON.parse` on the way out. Both are synchronous, so avoid them in hot paths. For anything structured or large, use IndexedDB.

---

## 8. BOM vs. DOM

| | BOM | DOM |
|---|---|---|
| Represents | The browser window environment | The HTML page content |
| Root | `window` | `document` (itself a property of `window`) |
| Standardisation | Historically ad-hoc; now largely covered by WHATWG HTML | W3C / WHATWG standardised |
| Example use | Redirect, read the URL, check connectivity | Change text, styling, add elements |

The line is blurrier than the table suggests — `document` is a `window` property, and `localStorage` and `fetch` sit on `window` without belonging to either model cleanly. "BOM" is a useful teaching category more than a formal spec boundary.

---

## References

- [MDN — Window](https://developer.mozilla.org/en-US/docs/Web/API/Window)
- [MDN — Location](https://developer.mozilla.org/en-US/docs/Web/API/Location)
- [MDN — Navigator](https://developer.mozilla.org/en-US/docs/Web/API/Navigator)
- [MDN — History API](https://developer.mozilla.org/en-US/docs/Web/API/History_API)
- [MDN — URLSearchParams](https://developer.mozilla.org/en-US/docs/Web/API/URLSearchParams)
- [MDN — Web Storage API](https://developer.mozilla.org/en-US/docs/Web/API/Web_Storage_API)